# Compute Authorship Embedding Similarity

Per-authorship cosine similarity between author mean embedding and individual work embedding.
Used as a content-fit signal in the AER pipeline for merge/split detection.

- **Source**: `author_embeddings` (106.7M, 1024-dim effective), `work_embeddings_v2` (414M, 1024-dim)
- **Method**: Cosine similarity per (author_id, work_id) pair
- **Output**: One row per authorship in Databricks
- **Job**: #105.9 aer-authorship-similarity

In [ ]:
# Configuration
OUTPUT_TABLE = "openalex.aer.authorship_embedding_similarity"
AUTHOR_EMBEDDINGS_TABLE = "openalex.aer.author_embeddings"
WORK_EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
WORKS_TABLE = "openalex.works.openalex_works"
EMBEDDING_DIM = 1024  # effective dim (author table has 1536 but 1024-1535 are NULL)
N_BATCHES = 5

## Setup output table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (
        work_id STRING,
        author_id BIGINT,
        cosine_similarity DOUBLE
    )
""")

existing = spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']
print(f"Output table has {existing:,} rows already computed")

## Phase 1: Single-work authors (shortcut)

Authors with `work_count=1` have mean embedding == their one paper's embedding,
so cosine_similarity is definitionally 1.0. No vector math needed.

In [ ]:
import time
from datetime import datetime

# Check if Phase 1 already done
single_work_count = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {AUTHOR_EMBEDDINGS_TABLE} WHERE work_count = 1
""").collect()[0]['n']

single_work_done = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {OUTPUT_TABLE} s
    JOIN {AUTHOR_EMBEDDINGS_TABLE} ae ON s.author_id = ae.author_id
    WHERE ae.work_count = 1
""").collect()[0]['n']

print(f"Single-work authors: {single_work_count:,} total, {single_work_done:,} already done")

if single_work_done >= single_work_count:
    print("Phase 1 already complete, skipping.")
else:
    phase1_start = time.time()
    spark.sql(f"""
        INSERT INTO {OUTPUT_TABLE}
        WITH single_authors AS (
            SELECT author_id
            FROM {AUTHOR_EMBEDDINGS_TABLE}
            WHERE work_count = 1
              AND author_id NOT IN (SELECT DISTINCT author_id FROM {OUTPUT_TABLE})
        ),
        authorship_pairs AS (
            SELECT
                CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
                CAST(w.id AS STRING) AS work_id
            FROM {WORKS_TABLE} w
            LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
            WHERE a.author.id IS NOT NULL
        )
        SELECT
            ap.work_id,
            ap.author_id,
            1.0 AS cosine_similarity
        FROM authorship_pairs ap
        JOIN single_authors sa ON ap.author_id = sa.author_id
    """)
    phase1_elapsed = time.time() - phase1_start
    phase1_count = spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']
    print(f"Phase 1 done: {phase1_count:,} rows in {phase1_elapsed/60:.1f}min")

## Phase 2: Multi-work authors (batched cosine similarity)

5 batches by `author_id % N_BATCHES`. Each batch joins work_embeddings with
author_embeddings and computes cosine similarity per authorship.

In [ ]:
def get_count():
    return spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']

def process_batch(batch_num):
    """Compute cosine similarity for multi-work authors where author_id % N_BATCHES == batch_num."""
    spark.sql(f"""
        INSERT INTO {OUTPUT_TABLE}
        WITH author_work_pairs AS (
            SELECT
                CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
                e.work_id,
                e.embedding AS work_embedding
            FROM {WORK_EMBEDDINGS_TABLE} e
            JOIN {WORKS_TABLE} w ON e.work_id = CAST(w.id AS STRING)
            LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
            WHERE a.author.id IS NOT NULL
              AND e.embedding IS NOT NULL
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) % {N_BATCHES} = {batch_num}
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) NOT IN (
                  SELECT author_id FROM {OUTPUT_TABLE}
                  WHERE author_id % {N_BATCHES} = {batch_num}
              )
        )
        SELECT
            awp.work_id,
            awp.author_id,
            aggregate(
                transform(
                    slice(ae.embedding, 1, {EMBEDDING_DIM}),
                    (v, i) -> v * awp.work_embedding[i]
                ),
                0.0D, (acc, v) -> acc + v
            ) / (
                sqrt(aggregate(
                    slice(ae.embedding, 1, {EMBEDDING_DIM}),
                    0.0D, (acc, v) -> acc + v * v
                )) *
                sqrt(aggregate(
                    awp.work_embedding,
                    0.0D, (acc, v) -> acc + v * v
                ))
            ) AS cosine_similarity
        FROM author_work_pairs awp
        JOIN {AUTHOR_EMBEDDINGS_TABLE} ae ON awp.author_id = ae.author_id
        WHERE ae.work_count > 1
    """)


start_time = time.time()
start_count = get_count()
remaining_est = 652_000_000  # ~652M multi-work authorship rows

print(f"Phase 2: Processing {N_BATCHES} batches for multi-work authors")
print(f"Starting from {start_count:,} rows (includes Phase 1)")
print("=" * 70)

for batch_num in range(N_BATCHES):
    batch_start = time.time()
    try:
        process_batch(batch_num)
        batch_elapsed = time.time() - batch_start

        current = get_count()
        added = current - start_count
        total_elapsed = time.time() - start_time
        rate = added / total_elapsed if total_elapsed > 0 else 0
        remaining_n = remaining_est - added if remaining_est > added else 0
        eta_min = remaining_n / rate / 60 if rate > 0 else 0

        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"Batch {batch_num + 1}/{N_BATCHES}: {batch_elapsed/60:.1f}min | "
            f"Total: {current:,} | "
            f"+{added:,} this phase | "
            f"Rate: {rate:.0f}/s | "
            f"ETA: {eta_min:.0f}min"
        )
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ERROR batch {batch_num}: {e}")
        print("Waiting 60s before continuing...")
        time.sleep(60)

print("=" * 70)
final = get_count()
total_time = (time.time() - start_time) / 60
print(f"Phase 2 done! {final:,} total rows in {total_time:.1f}min")

## Verify results

In [ ]:
spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT author_id) AS unique_authors,
        COUNT(DISTINCT work_id) AS unique_works,
        ROUND(AVG(cosine_similarity), 4) AS avg_sim,
        ROUND(PERCENTILE(cosine_similarity, 0.05), 4) AS p5_sim,
        ROUND(PERCENTILE(cosine_similarity, 0.50), 4) AS median_sim,
        ROUND(PERCENTILE(cosine_similarity, 0.95), 4) AS p95_sim,
        ROUND(MIN(cosine_similarity), 4) AS min_sim,
        ROUND(MAX(cosine_similarity), 4) AS max_sim,
        SUM(CASE WHEN cosine_similarity IS NULL THEN 1 ELSE 0 END) AS null_count,
        SUM(CASE WHEN cosine_similarity = 1.0 THEN 1 ELSE 0 END) AS exact_one_count
    FROM {OUTPUT_TABLE}
""").show(truncate=False)

In [ ]:
# Verify single-work count matches
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {AUTHOR_EMBEDDINGS_TABLE} WHERE work_count = 1) AS expected_single_work,
        (SELECT SUM(CAST(work_count AS BIGINT)) FROM {AUTHOR_EMBEDDINGS_TABLE}) AS expected_total_rows
""").show(truncate=False)

In [ ]:
# Spot check: low similarity works (potential overmerge candidates)
spark.sql(f"""
    SELECT
        s.author_id,
        s.work_id,
        ROUND(s.cosine_similarity, 4) AS sim,
        ae.work_count
    FROM {OUTPUT_TABLE} s
    JOIN {AUTHOR_EMBEDDINGS_TABLE} ae ON s.author_id = ae.author_id
    WHERE s.cosine_similarity < 0.3
      AND ae.work_count >= 10
    ORDER BY s.cosine_similarity ASC
    LIMIT 20
""").show(truncate=False)

## Optimize table

In [ ]:
spark.sql(f"OPTIMIZE {OUTPUT_TABLE}")
print(f"Table optimized: {OUTPUT_TABLE}")